# Smart Document Scanner — Final Experiments Notebook (FIXED)

**Project:** Confidence-Guided Hybrid Document Scanning with Shadow-Aware Enhancement  
**Author:** Valeriia Boichenko  
**Purpose:** Run baselines, proposed method, collect numerical results, save comparison figures and CSV files for the IEEE final report.

This corrected version works safely with ZIP archives created on macOS. It automatically removes `__MACOSX` folders and hidden files such as `._photo.jpeg`, because OpenCV cannot read them as images.

The notebook can run in two modes:

1. **User dataset mode:** upload your own document photos as a `.zip` file. This is the recommended mode for the final report.
2. **Demo mode:** automatically creates synthetic smartphone-like document photos with perspective distortion, shadows, blur and background clutter.

For the final paper, use real images if possible. Demo mode is useful only for checking that the whole pipeline works.


In [ ]:
# ============================================================
# 1. Install / import dependencies
# ============================================================

import os
import cv2
import math
import time
import shutil
import zipfile
import random
import warnings
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
from PIL import Image, ImageDraw, ImageFont

# Install scikit-image if it is missing.
try:
    from skimage.metrics import structural_similarity as ssim
    from skimage.metrics import peak_signal_noise_ratio as psnr
except Exception:
    !pip -q install scikit-image
    from skimage.metrics import structural_similarity as ssim
    from skimage.metrics import peak_signal_noise_ratio as psnr

# Optional OCR support.
# This is useful for the final paper because it gives an OCR confidence metric.
# It may take about 1 minute in Google Colab.
INSTALL_TESSERACT = True

if INSTALL_TESSERACT:
    try:
        import pytesseract
        OCR_AVAILABLE = True
    except Exception:
        !sudo apt-get -qq update
        !sudo apt-get -qq install -y tesseract-ocr
        !pip -q install pytesseract
        try:
            import pytesseract
            OCR_AVAILABLE = True
        except Exception:
            OCR_AVAILABLE = False
else:
    try:
        import pytesseract
        OCR_AVAILABLE = True
    except Exception:
        OCR_AVAILABLE = False

warnings.filterwarnings("ignore")

print("OpenCV:", cv2.__version__)
print("OCR available:", OCR_AVAILABLE)


In [ ]:
# ============================================================
# 2. Project configuration
# ============================================================

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_DIR = Path("/content/smart_document_scanner")
INPUT_DIR = PROJECT_DIR / "input_images"
SYNTH_DIR = PROJECT_DIR / "synthetic_dataset"
RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"
OUTPUTS_DIR = RESULTS_DIR / "processed_outputs"
TABLES_DIR = RESULTS_DIR / "tables"

for p in [PROJECT_DIR, INPUT_DIR, SYNTH_DIR, RESULTS_DIR, FIGURES_DIR, OUTPUTS_DIR, TABLES_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Final report mode:
# True  = upload your own real document photos as ZIP or individual images.
# False = run synthetic demo dataset.
USE_UPLOADED_IMAGES = True

# If True, the upload folder is cleaned before extracting your new ZIP.
# This prevents old files and macOS hidden files from staying in the experiment.
RESET_INPUT_FOLDER_BEFORE_UPLOAD = True

# Number of synthetic test images created in demo mode.
N_SYNTHETIC_IMAGES = 12

# Standard rectified document size.
WARP_W, WARP_H = 900, 1200

# Confidence-score weights from the proposal.
WA, WR, WE = 0.30, 0.30, 0.40
CONF_THRESHOLD = 0.45

# Supported image extensions.
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

print("Project directory:", PROJECT_DIR)
print("Use uploaded images:", USE_UPLOADED_IMAGES)


## 3. Upload your own dataset

For real experiments, prepare a ZIP file with document photos:

```text
dataset.zip
  img_001.jpg
  img_002.jpg
  img_003.png
  ...
```

This corrected notebook also works if your ZIP was created on macOS. It automatically deletes:

```text
__MACOSX/
._photo.jpeg
.DS_Store
```

These are hidden service files, not real images, and they cause `cv2.imread()` errors.

If you want to use the synthetic demo dataset instead, set `USE_UPLOADED_IMAGES = False` in the configuration cell above.


In [ ]:
# ============================================================
# 3. Upload / extract dataset safely
# ============================================================


def clean_macos_hidden_files(root_dir):
    """Remove macOS service folders/files that break cv2.imread()."""
    root_dir = Path(root_dir)

    for macosx_dir in list(root_dir.rglob("__MACOSX")):
        print("Deleting macOS folder:", macosx_dir)
        shutil.rmtree(macosx_dir, ignore_errors=True)

    for hidden_file in list(root_dir.rglob("._*")):
        print("Deleting hidden macOS file:", hidden_file)
        hidden_file.unlink(missing_ok=True)

    for ds_store in list(root_dir.rglob(".DS_Store")):
        print("Deleting .DS_Store:", ds_store)
        ds_store.unlink(missing_ok=True)


def safe_extract_zip(zip_path, extract_to):
    """Extract a ZIP archive and clean macOS hidden files after extraction."""
    zip_path = Path(zip_path)
    extract_to = Path(extract_to)
    extract_to.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(extract_to)

    clean_macos_hidden_files(extract_to)
    print("Extracted and cleaned:", zip_path.name, "->", extract_to)


if USE_UPLOADED_IMAGES:
    from google.colab import files

    if RESET_INPUT_FOLDER_BEFORE_UPLOAD and INPUT_DIR.exists():
        shutil.rmtree(INPUT_DIR, ignore_errors=True)
        INPUT_DIR.mkdir(parents=True, exist_ok=True)
        print("Cleaned input folder:", INPUT_DIR)

    uploaded = files.upload()

    for filename in uploaded.keys():
        file_path = Path(filename)

        if file_path.suffix.lower() == ".zip":
            safe_extract_zip(file_path, INPUT_DIR)

        elif file_path.suffix.lower() in IMAGE_EXTENSIONS:
            destination = INPUT_DIR / file_path.name
            shutil.copy(file_path, destination)
            print("Copied image:", destination)

        else:
            print("Skipped unsupported uploaded file:", filename)

    clean_macos_hidden_files(INPUT_DIR)
else:
    print("Upload skipped because USE_UPLOADED_IMAGES = False")


## 4. Synthetic document photo generation

This section creates realistic-looking document photos for testing:

- perspective distortion;
- uneven illumination;
- artificial shadows;
- blur;
- background clutter;
- known ground-truth document polygon.

This is useful because it gives us approximate ground truth for IoU, PSNR and SSIM.

In [ ]:
# ============================================================
# 4. Synthetic document generator
# ============================================================

def create_clean_document(width=WARP_W, height=WARP_H, idx=0):
    """Create a clean white document with text, table-like lines and a small diagram."""
    img = Image.new("RGB", (width, height), (245, 245, 240))
    draw = ImageDraw.Draw(img)

    try:
        title_font = ImageFont.truetype("DejaVuSans-Bold.ttf", 42)
        head_font = ImageFont.truetype("DejaVuSans-Bold.ttf", 26)
        body_font = ImageFont.truetype("DejaVuSans.ttf", 22)
        small_font = ImageFont.truetype("DejaVuSans.ttf", 18)
    except Exception:
        title_font = head_font = body_font = small_font = None

    margin = 70
    draw.text((margin, 50), f"Smart Document Scanner Sample {idx+1}", fill=(25,25,25), font=title_font)
    draw.line((margin, 110, width-margin, 110), fill=(40,40,40), width=3)

    y = 145
    sections = [
        ("Problem", "Mobile document images often contain perspective distortion, blur, shadows and weak borders."),
        ("Method", "The proposed method combines contour and Hough candidates using an explainable confidence score."),
        ("Enhancement", "Lab luminance normalization is applied before adaptive thresholding and CLAHE enhancement."),
    ]

    for title, text in sections:
        draw.text((margin, y), title, fill=(30,30,30), font=head_font)
        y += 38
        words = text.split()
        line = ""
        for w in words:
            if len(line + " " + w) < 70:
                line += " " + w
            else:
                draw.text((margin, y), line.strip(), fill=(45,45,45), font=body_font)
                y += 32
                line = w
        if line:
            draw.text((margin, y), line.strip(), fill=(45,45,45), font=body_font)
            y += 48

    # Table
    table_x, table_y = margin, y + 15
    table_w, row_h = width - 2*margin, 55
    cols = [0, 260, 510, table_w]
    rows = ["Method", "Detection", "Enhancement", "Runtime"]
    for r in range(5):
        draw.rectangle((table_x, table_y + r*row_h, table_x + table_w, table_y + (r+1)*row_h), outline=(80,80,80), width=2)
    for c in cols[1:-1]:
        draw.line((table_x+c, table_y, table_x+c, table_y+5*row_h), fill=(80,80,80), width=2)
    for i, h in enumerate(["Method", "Page detection", "Lighting", "Output"]):
        draw.text((table_x + cols[i] + 10, table_y + 15), h, fill=(20,20,20), font=small_font)

    table_rows = [
        ["A", "Contours", "None", "B/W"],
        ["B", "Hough", "None", "B/W"],
        ["C", "Adaptive mask", "Partial", "B/W"],
        ["Proposed", "Hybrid", "Lab norm", "B/W + Color"],
    ]
    for r, row in enumerate(table_rows):
        for i, val in enumerate(row):
            draw.text((table_x + cols[i] + 10, table_y + (r+1)*row_h + 15), val, fill=(35,35,35), font=small_font)

    # Small geometric diagram
    diagram_y = table_y + 5*row_h + 60
    draw.rectangle((margin, diagram_y, margin+230, diagram_y+150), outline=(50,50,50), width=3)
    draw.polygon([(margin+330, diagram_y+25), (margin+560, diagram_y), (margin+530, diagram_y+155), (margin+300, diagram_y+140)], outline=(50,50,50), fill=None)
    draw.text((margin, diagram_y+180), "Figure: perspective correction by homography", fill=(50,50,50), font=small_font)

    return np.array(img)


def random_perspective_points(canvas_w, canvas_h):
    """Generate a random quadrilateral where the document will be placed."""
    margin_x = random.randint(80, 180)
    margin_y = random.randint(60, 160)

    tl = [margin_x + random.randint(-30, 40), margin_y + random.randint(-20, 50)]
    tr = [canvas_w - margin_x + random.randint(-40, 30), margin_y + random.randint(-20, 70)]
    br = [canvas_w - margin_x + random.randint(-50, 30), canvas_h - margin_y + random.randint(-70, 20)]
    bl = [margin_x + random.randint(-30, 50), canvas_h - margin_y + random.randint(-50, 30)]
    return np.float32([tl, tr, br, bl])


def add_shadow_and_noise(canvas):
    """Add smooth illumination variation, shadows, noise and optional blur."""
    h, w = canvas.shape[:2]
    result = canvas.astype(np.float32)

    # Smooth illumination gradient
    x = np.linspace(0.75, 1.15, w).reshape(1, w, 1)
    y = np.linspace(0.85, 1.10, h).reshape(h, 1, 1)
    result *= x * y

    # Elliptical shadow
    shadow = np.ones((h, w), dtype=np.float32)
    cx, cy = random.randint(w//4, 3*w//4), random.randint(h//4, 3*h//4)
    rx, ry = random.randint(w//4, w//2), random.randint(h//6, h//3)
    yy, xx = np.mgrid[0:h, 0:w]
    ellipse = ((xx-cx)**2)/(rx**2) + ((yy-cy)**2)/(ry**2)
    shadow_strength = random.uniform(0.45, 0.75)
    shadow[ellipse < 1] *= shadow_strength
    shadow = cv2.GaussianBlur(shadow, (151,151), 0)
    result *= shadow[..., None]

    # Noise
    noise = np.random.normal(0, random.uniform(2, 7), result.shape)
    result += noise

    result = np.clip(result, 0, 255).astype(np.uint8)

    # Occasional blur
    if random.random() < 0.5:
        result = cv2.GaussianBlur(result, (3,3), 0)

    return result


def create_synthetic_photo(idx, out_dir=SYNTH_DIR):
    """Create one smartphone-like document photo and save input, ground truth and polygon mask."""
    clean_doc = create_clean_document(idx=idx)

    canvas_h, canvas_w = 1500, 1200
    bg_color = np.array([random.randint(120, 180), random.randint(110, 165), random.randint(95, 150)], dtype=np.uint8)
    canvas = np.ones((canvas_h, canvas_w, 3), dtype=np.uint8) * bg_color

    # Add background texture
    texture = np.random.normal(0, 10, canvas.shape).astype(np.int16)
    canvas = np.clip(canvas.astype(np.int16) + texture, 0, 255).astype(np.uint8)

    # Add random background lines
    for _ in range(random.randint(3, 8)):
        p1 = (random.randint(0, canvas_w), random.randint(0, canvas_h))
        p2 = (random.randint(0, canvas_w), random.randint(0, canvas_h))
        color = tuple(int(x) for x in np.random.randint(80, 180, size=3))
        cv2.line(canvas, p1, p2, color, random.randint(1, 3))

    src = np.float32([[0,0], [WARP_W-1,0], [WARP_W-1,WARP_H-1], [0,WARP_H-1]])
    dst = random_perspective_points(canvas_w, canvas_h)
    H = cv2.getPerspectiveTransform(src, dst)

    warped_doc = cv2.warpPerspective(clean_doc, H, (canvas_w, canvas_h))
    mask = cv2.warpPerspective(np.ones((WARP_H, WARP_W), dtype=np.uint8)*255, H, (canvas_w, canvas_h))

    # Composite document on background
    mask3 = mask[..., None] / 255.0
    photo = (warped_doc * mask3 + canvas * (1 - mask3)).astype(np.uint8)

    photo = add_shadow_and_noise(photo)

    name = f"synthetic_{idx+1:03d}"
    cv2.imwrite(str(out_dir / f"{name}.jpg"), cv2.cvtColor(photo, cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(out_dir / f"{name}_gt_clean.png"), cv2.cvtColor(clean_doc, cv2.COLOR_RGB2BGR))
    cv2.imwrite(str(out_dir / f"{name}_gt_mask.png"), mask)

    meta = {
        "image": str(out_dir / f"{name}.jpg"),
        "gt_clean": str(out_dir / f"{name}_gt_clean.png"),
        "gt_mask": str(out_dir / f"{name}_gt_mask.png"),
        "quad": dst.tolist()
    }
    return meta


if not USE_UPLOADED_IMAGES:
    # Clean old synthetic files
    for f in SYNTH_DIR.glob("*"):
        f.unlink()

    synthetic_meta = [create_synthetic_photo(i) for i in range(N_SYNTHETIC_IMAGES)]
    pd.DataFrame(synthetic_meta).to_json(SYNTH_DIR / "synthetic_metadata.json", orient="records", indent=2)

    print(f"Created {len(synthetic_meta)} synthetic document photos in:", SYNTH_DIR)

In [ ]:
# ============================================================
# 5. Utility functions
# ============================================================

def read_rgb(path):
    img = cv2.imread(str(path))
    if img is None:
        raise ValueError(f"Could not read image: {path}")
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)


def save_rgb(path, img):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(path), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))


def order_points(pts):
    """Order points as top-left, top-right, bottom-right, bottom-left."""
    pts = np.array(pts, dtype=np.float32)
    rect = np.zeros((4, 2), dtype=np.float32)

    s = pts.sum(axis=1)
    diff = np.diff(pts, axis=1).reshape(-1)

    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]
    return rect


def four_point_transform(image, pts, width=WARP_W, height=WARP_H):
    rect = order_points(pts)
    dst = np.array([[0,0], [width-1,0], [width-1,height-1], [0,height-1]], dtype=np.float32)
    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (width, height))
    return warped


def polygon_mask(shape, quad):
    mask = np.zeros(shape[:2], dtype=np.uint8)
    if quad is not None:
        pts = np.array(order_points(quad), dtype=np.int32)
        cv2.fillPoly(mask, [pts], 255)
    return mask


def safe_canny(gray):
    med = np.median(gray)
    lower = int(max(0, 0.66 * med))
    upper = int(min(255, 1.33 * med))
    return cv2.Canny(gray, lower, upper)


def preprocess_for_detection(image):
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    return gray


def largest_quad_from_contours(contours, image_area, min_area_ratio=0.05):
    candidates = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        if area < min_area_ratio * image_area:
            continue
        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, 0.02 * peri, True)

        if len(approx) == 4:
            quad = approx.reshape(4, 2)
        else:
            rect = cv2.minAreaRect(cnt)
            quad = cv2.boxPoints(rect)

        candidates.append((area, quad))

    if not candidates:
        return None

    candidates.sort(key=lambda x: x[0], reverse=True)
    return order_points(candidates[0][1])


def internal_angles(quad):
    q = order_points(quad)
    angles = []
    for i in range(4):
        p_prev = q[(i-1) % 4]
        p = q[i]
        p_next = q[(i+1) % 4]
        v1 = p_prev - p
        v2 = p_next - p
        denom = (np.linalg.norm(v1) * np.linalg.norm(v2)) + 1e-8
        cosang = np.clip(np.dot(v1, v2) / denom, -1.0, 1.0)
        angles.append(np.degrees(np.arccos(cosang)))
    return angles


def rectangularity_score(quad):
    angles = internal_angles(quad)
    score = 1.0 - sum(abs(a - 90.0) for a in angles) / (4.0 * 90.0)
    return float(np.clip(score, 0, 1))


def edge_support_score(gray, quad):
    edges = safe_canny(gray)
    q = order_points(quad).astype(np.int32)
    supports = []

    for i in range(4):
        p1 = tuple(q[i])
        p2 = tuple(q[(i+1) % 4])

        line_mask = np.zeros_like(edges)
        cv2.line(line_mask, p1, p2, 255, thickness=5)

        total = np.count_nonzero(line_mask)
        support = np.count_nonzero(cv2.bitwise_and(edges, line_mask)) / (total + 1e-8)
        supports.append(support)

    # Normalize because a thin edge occupies only a small fraction of the thick line mask.
    score = min(1.0, float(np.mean(supports) * 12.0))
    return score


def confidence_score(image, quad, wa=WA, wr=WR, we=WE):
    if quad is None:
        return 0.0, {"area_score": 0, "rect_score": 0, "edge_score": 0}

    h, w = image.shape[:2]
    image_area = h * w
    quad_area = abs(cv2.contourArea(order_points(quad).astype(np.float32)))
    area_ratio = quad_area / (image_area + 1e-8)
    area_score = min(1.0, area_ratio / 0.7)

    gray = preprocess_for_detection(image)
    rect_score = rectangularity_score(quad)
    edge_score = edge_support_score(gray, quad)

    c = wa * area_score + wr * rect_score + we * edge_score
    details = {
        "area_score": float(area_score),
        "rect_score": float(rect_score),
        "edge_score": float(edge_score)
    }
    return float(np.clip(c, 0, 1)), details


def iou_masks(mask1, mask2):
    m1 = mask1 > 0
    m2 = mask2 > 0
    inter = np.logical_and(m1, m2).sum()
    union = np.logical_or(m1, m2).sum()
    return float(inter / (union + 1e-8))


def background_uniformity(gray_or_rgb):
    if len(gray_or_rgb.shape) == 3:
        gray = cv2.cvtColor(gray_or_rgb, cv2.COLOR_RGB2GRAY)
    else:
        gray = gray_or_rgb
    # Bright pixels are treated as page background approximation.
    thresh = np.percentile(gray, 70)
    bg = gray[gray >= thresh]
    if len(bg) < 10:
        return float(np.std(gray))
    return float(np.std(bg))


def text_background_contrast(gray_or_rgb):
    if len(gray_or_rgb.shape) == 3:
        gray = cv2.cvtColor(gray_or_rgb, cv2.COLOR_RGB2GRAY)
    else:
        gray = gray_or_rgb
    dark = gray[gray <= np.percentile(gray, 20)]
    bright = gray[gray >= np.percentile(gray, 80)]
    return float(np.mean(bright) - np.mean(dark))


def ocr_confidence(img):
    if not OCR_AVAILABLE:
        return np.nan

    if len(img.shape) == 3:
        gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    else:
        gray = img

    try:
        data = pytesseract.image_to_data(gray, output_type=pytesseract.Output.DATAFRAME)
        conf = pd.to_numeric(data["conf"], errors="coerce")
        conf = conf[(conf >= 0) & (conf <= 100)]
        if len(conf) == 0:
            return np.nan
        return float(conf.mean())
    except Exception:
        return np.nan

In [ ]:
# ============================================================
# 6. Page detection methods
# ============================================================

def detect_contour_quad(image):
    """Baseline A detector: Canny -> contours -> largest quadrilateral."""
    gray = preprocess_for_detection(image)
    edges = safe_canny(gray)
    edges = cv2.dilate(edges, np.ones((3,3), np.uint8), iterations=1)

    contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    quad = largest_quad_from_contours(contours, image.shape[0] * image.shape[1])
    return quad, {"edges": edges}


def detect_hough_quad(image):
    """Baseline B detector: Canny -> HoughLinesP -> min-area rectangle around line points."""
    gray = preprocess_for_detection(image)
    edges = safe_canny(gray)

    lines = cv2.HoughLinesP(
        edges, rho=1, theta=np.pi/180, threshold=80,
        minLineLength=max(image.shape[:2]) // 6,
        maxLineGap=40
    )

    if lines is None or len(lines) < 4:
        return None, {"edges": edges, "lines": None}

    pts = []
    for line in lines[:, 0, :]:
        x1, y1, x2, y2 = line
        pts.append([x1, y1])
        pts.append([x2, y2])

    pts = np.array(pts, dtype=np.float32)
    rect = cv2.minAreaRect(pts)
    quad = cv2.boxPoints(rect)

    return order_points(quad), {"edges": edges, "lines": lines}


def detect_adaptive_mask_quad(image):
    """Baseline C detector: adaptive threshold mask -> morphology -> contours."""
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    blur = cv2.GaussianBlur(gray, (5,5), 0)

    mask = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        51, 7
    )

    # We want a large page-like connected component.
    mask_inv = 255 - mask
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (15,15))
    morph = cv2.morphologyEx(mask_inv, cv2.MORPH_CLOSE, kernel, iterations=2)
    morph = cv2.dilate(morph, kernel, iterations=1)

    contours, _ = cv2.findContours(morph, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    quad = largest_quad_from_contours(contours, image.shape[0] * image.shape[1], min_area_ratio=0.04)

    return quad, {"mask": mask, "morph": morph}


def detect_proposed_quad(image):
    """Proposed detector: contour candidate + Hough candidate + confidence selection."""
    quad_a, dbg_a = detect_contour_quad(image)
    quad_b, dbg_b = detect_hough_quad(image)

    score_a, details_a = confidence_score(image, quad_a)
    score_b, details_b = confidence_score(image, quad_b)

    candidates = [
        ("contour_candidate", quad_a, score_a, details_a),
        ("hough_candidate", quad_b, score_b, details_b)
    ]
    candidates = [c for c in candidates if c[1] is not None]

    if not candidates:
        return None, {"selected": None, "score": 0, "candidates": []}

    selected = max(candidates, key=lambda x: x[2])

    # Safe fallback if confidence is very low.
    if selected[2] < CONF_THRESHOLD:
        gray = preprocess_for_detection(image)
        edges = safe_canny(gray)
        contours, _ = cv2.findContours(edges, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        fallback = largest_quad_from_contours(contours, image.shape[0] * image.shape[1], min_area_ratio=0.02)
        if fallback is not None:
            selected = ("fallback_min_area", fallback, selected[2], selected[3])

    return selected[1], {
        "selected": selected[0],
        "score": selected[2],
        "candidates": candidates,
        "contour_debug": dbg_a,
        "hough_debug": dbg_b
    }

In [ ]:
# ============================================================
# 7. Enhancement methods and full pipelines
# ============================================================

def otsu_bw(warped_rgb):
    gray = cv2.cvtColor(warped_rgb, cv2.COLOR_RGB2GRAY)
    _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return bw


def adaptive_bw(warped_rgb_or_gray):
    if len(warped_rgb_or_gray.shape) == 3:
        gray = cv2.cvtColor(warped_rgb_or_gray, cv2.COLOR_RGB2GRAY)
    else:
        gray = warped_rgb_or_gray
    bw = cv2.adaptiveThreshold(
        gray, 255,
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY,
        31, 11
    )
    return bw


def lab_shadow_normalization(warped_rgb, kernel_size=61):
    """Shadow-aware enhancement: normalize Lab L channel with large-kernel illumination estimate."""
    lab = cv2.cvtColor(warped_rgb, cv2.COLOR_RGB2LAB)
    L, A, B = cv2.split(lab)

    if kernel_size % 2 == 0:
        kernel_size += 1

    illumination = cv2.GaussianBlur(L, (kernel_size, kernel_size), 0)
    illumination = illumination.astype(np.float32) + 1.0

    normalized = (L.astype(np.float32) / illumination) * np.mean(illumination)
    normalized = np.clip(normalized, 0, 255).astype(np.uint8)

    lab_norm = cv2.merge([normalized, A, B])
    rgb_norm = cv2.cvtColor(lab_norm, cv2.COLOR_LAB2RGB)

    return rgb_norm, normalized, illumination.astype(np.uint8)


def color_scan_output(warped_rgb):
    normalized_rgb, L_norm, illum = lab_shadow_normalization(warped_rgb)

    lab = cv2.cvtColor(normalized_rgb, cv2.COLOR_RGB2LAB)
    L, A, B = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    L2 = clahe.apply(L)

    enhanced = cv2.cvtColor(cv2.merge([L2, A, B]), cv2.COLOR_LAB2RGB)

    # Gentle sharpening
    blur = cv2.GaussianBlur(enhanced, (0,0), 1.0)
    sharp = cv2.addWeighted(enhanced, 1.25, blur, -0.25, 0)
    return sharp, L_norm, illum


def run_pipeline(image, method_name):
    """Run one of the four methods and return detected quad, warped page, output image and debug information."""
    t0 = time.perf_counter()

    if method_name == "Baseline A: Contour + Otsu":
        quad, debug = detect_contour_quad(image)
    elif method_name == "Baseline B: Hough + Adaptive":
        quad, debug = detect_hough_quad(image)
    elif method_name == "Baseline C: Adaptive Mask + Adaptive":
        quad, debug = detect_adaptive_mask_quad(image)
    elif method_name == "Proposed: Hybrid + Lab Norm":
        quad, debug = detect_proposed_quad(image)
    else:
        raise ValueError("Unknown method: " + method_name)

    if quad is None:
        runtime_ms = (time.perf_counter() - t0) * 1000
        return {
            "quad": None,
            "warped": None,
            "output_bw": None,
            "output_color": None,
            "runtime_ms": runtime_ms,
            "debug": debug,
            "success": False
        }

    warped = four_point_transform(image, quad)

    if method_name == "Baseline A: Contour + Otsu":
        output_bw = otsu_bw(warped)
        output_color = warped
    elif method_name in ["Baseline B: Hough + Adaptive", "Baseline C: Adaptive Mask + Adaptive"]:
        output_bw = adaptive_bw(warped)
        output_color = warped
    else:
        color_out, L_norm, illum = color_scan_output(warped)
        output_color = color_out
        output_bw = adaptive_bw(L_norm)
        debug["L_norm"] = L_norm
        debug["illumination"] = illum

    runtime_ms = (time.perf_counter() - t0) * 1000

    return {
        "quad": quad,
        "warped": warped,
        "output_bw": output_bw,
        "output_color": output_color,
        "runtime_ms": runtime_ms,
        "debug": debug,
        "success": True
    }


METHODS = [
    "Baseline A: Contour + Otsu",
    "Baseline B: Hough + Adaptive",
    "Baseline C: Adaptive Mask + Adaptive",
    "Proposed: Hybrid + Lab Norm"
]

In [ ]:
# ============================================================
# 8. Prepare image list safely
# ============================================================


def collect_valid_image_paths(root_dir):
    """Collect only real images that OpenCV can read."""
    root_dir = Path(root_dir)
    clean_macos_hidden_files(root_dir)

    candidate_paths = []
    for p in root_dir.rglob("*"):
        if not p.is_file():
            continue
        if "__MACOSX" in str(p):
            continue
        if p.name.startswith("._") or p.name.startswith("."):
            continue
        if p.suffix.lower() not in IMAGE_EXTENSIONS:
            continue
        candidate_paths.append(p)

    valid_paths = []
    skipped_paths = []

    for path in sorted(candidate_paths):
        img = cv2.imread(str(path))
        if img is not None:
            valid_paths.append(path)
        else:
            skipped_paths.append(path)

    if skipped_paths:
        print("Skipped unreadable files:")
        for p in skipped_paths:
            print("  -", p)

    return valid_paths


if USE_UPLOADED_IMAGES:
    image_paths = collect_valid_image_paths(INPUT_DIR)
    metadata = None
else:
    image_paths = sorted(SYNTH_DIR.glob("synthetic_*.jpg"))
    # Exclude ground-truth files
    image_paths = [p for p in image_paths if "_gt_" not in p.name]
    metadata = pd.read_json(SYNTH_DIR / "synthetic_metadata.json")

print("Number of valid input images:", len(image_paths))
print("First images:", [p.name for p in image_paths[:5]])

if len(image_paths) == 0:
    raise RuntimeError(
        "No valid images found. Upload a ZIP with .jpg/.jpeg/.png files, "
        "or set USE_UPLOADED_IMAGES = False to use the demo dataset."
    )

# Show sample inputs
n_show = min(4, len(image_paths))
plt.figure(figsize=(16, 5))
for i, path in enumerate(image_paths[:n_show]):
    img = read_rgb(path)
    plt.subplot(1, n_show, i+1)
    plt.imshow(img)
    plt.title(path.name)
    plt.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 9. Run all experiments
# ============================================================

all_rows = []

# Clean old outputs
for folder in [FIGURES_DIR, OUTPUTS_DIR, TABLES_DIR]:
    for f in folder.glob("*"):
        if f.is_file():
            f.unlink()

for idx, image_path in enumerate(image_paths):
    img = read_rgb(image_path)
    stem = image_path.stem

    gt_mask = None
    gt_clean = None
    true_quad = None

    if not USE_UPLOADED_IMAGES:
        gt_mask_path = SYNTH_DIR / f"{stem}_gt_mask.png"
        gt_clean_path = SYNTH_DIR / f"{stem}_gt_clean.png"
        if gt_mask_path.exists():
            gt_mask = cv2.imread(str(gt_mask_path), cv2.IMREAD_GRAYSCALE)
        if gt_clean_path.exists():
            gt_clean = read_rgb(gt_clean_path)
        try:
            true_quad = np.array(metadata.loc[metadata["image"].str.contains(image_path.name), "quad"].iloc[0], dtype=np.float32)
        except Exception:
            true_quad = None

    for method in METHODS:
        result = run_pipeline(img, method)

        # Save outputs
        safe_method_name = method.replace(":", "").replace("+", "plus").replace(" ", "_").replace("/", "_")
        if result["success"]:
            save_rgb(OUTPUTS_DIR / f"{stem}_{safe_method_name}_color.png", result["output_color"])
            cv2.imwrite(str(OUTPUTS_DIR / f"{stem}_{safe_method_name}_bw.png"), result["output_bw"])

        # Metrics
        pred_mask = polygon_mask(img.shape, result["quad"]) if result["quad"] is not None else np.zeros(img.shape[:2], np.uint8)

        detection_iou = np.nan
        if gt_mask is not None:
            detection_iou = iou_masks(gt_mask, pred_mask)

        psnr_value = np.nan
        ssim_value = np.nan
        if gt_clean is not None and result["success"]:
            # Compare color scan with clean document.
            color_resized = cv2.resize(result["output_color"], (gt_clean.shape[1], gt_clean.shape[0]))
            psnr_value = psnr(gt_clean, color_resized, data_range=255)
            ssim_value = ssim(
                cv2.cvtColor(gt_clean, cv2.COLOR_RGB2GRAY),
                cv2.cvtColor(color_resized, cv2.COLOR_RGB2GRAY),
                data_range=255
            )

        bg_std = np.nan
        contrast = np.nan
        ocr_conf = np.nan
        if result["success"]:
            bg_std = background_uniformity(result["output_bw"])
            contrast = text_background_contrast(result["output_bw"])
            ocr_conf = ocr_confidence(result["output_bw"])

        conf, conf_details = confidence_score(img, result["quad"]) if result["quad"] is not None else (0, {})

        all_rows.append({
            "image": image_path.name,
            "method": method,
            "detection_success": int(result["success"]),
            "detection_iou": detection_iou,
            "psnr": psnr_value,
            "ssim": ssim_value,
            "background_std_lower_better": bg_std,
            "text_background_contrast_higher_better": contrast,
            "ocr_confidence_higher_better": ocr_conf,
            "runtime_ms_lower_better": result["runtime_ms"],
            "confidence_score": conf,
            "area_score": conf_details.get("area_score", np.nan),
            "rectangularity_score": conf_details.get("rect_score", np.nan),
            "edge_support_score": conf_details.get("edge_score", np.nan)
        })

results_df = pd.DataFrame(all_rows)
results_csv = TABLES_DIR / "per_image_results.csv"
results_df.to_csv(results_csv, index=False)

print("Saved per-image results to:", results_csv)
display(results_df.head(12))

In [ ]:
# ============================================================
# 10. Method-level summary tables
# ============================================================

summary_df = (
    results_df
    .groupby("method")
    .agg(
        images=("image", "count"),
        detection_success_rate=("detection_success", "mean"),
        mean_detection_iou=("detection_iou", "mean"),
        mean_psnr=("psnr", "mean"),
        mean_ssim=("ssim", "mean"),
        mean_background_std=("background_std_lower_better", "mean"),
        mean_text_contrast=("text_background_contrast_higher_better", "mean"),
        mean_ocr_confidence=("ocr_confidence_higher_better", "mean"),
        mean_runtime_ms=("runtime_ms_lower_better", "mean"),
        mean_confidence_score=("confidence_score", "mean")
    )
    .reset_index()
)

summary_csv = TABLES_DIR / "method_summary.csv"
summary_df.to_csv(summary_csv, index=False)

print("Saved method summary to:", summary_csv)
display(summary_df)

In [ ]:
# ============================================================
# 11. Create visual comparison figures for the report
# ============================================================

def draw_quad_on_image(img, quad, color=(255,0,0)):
    out = img.copy()
    if quad is not None:
        pts = order_points(quad).astype(np.int32)
        cv2.polylines(out, [pts], isClosed=True, color=color, thickness=5)
    return out


def make_comparison_figure(image_path, save_path):
    img = read_rgb(image_path)
    method_results = [run_pipeline(img, m) for m in METHODS]

    fig, axes = plt.subplots(2, 5, figsize=(20, 9))

    # Row 1: detection quadrilaterals
    axes[0,0].imshow(img)
    axes[0,0].set_title("Input image")
    axes[0,0].axis("off")

    for j, (m, r) in enumerate(zip(METHODS, method_results), start=1):
        axes[0,j].imshow(draw_quad_on_image(img, r["quad"]))
        axes[0,j].set_title(m.replace(": ", "\n"))
        axes[0,j].axis("off")

    # Row 2: outputs
    axes[1,0].imshow(img)
    axes[1,0].set_title("Original")
    axes[1,0].axis("off")

    for j, (m, r) in enumerate(zip(METHODS, method_results), start=1):
        if r["success"]:
            if m == "Proposed: Hybrid + Lab Norm":
                axes[1,j].imshow(r["output_color"])
                axes[1,j].set_title("Proposed color scan")
            else:
                axes[1,j].imshow(r["output_bw"], cmap="gray")
                axes[1,j].set_title("B/W output")
        else:
            axes[1,j].text(0.5, 0.5, "Detection failed", ha="center", va="center")
            axes[1,j].set_title("Failed")
        axes[1,j].axis("off")

    plt.tight_layout()
    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close()


for p in image_paths[:min(3, len(image_paths))]:
    out_path = FIGURES_DIR / f"comparison_{p.stem}.png"
    make_comparison_figure(p, out_path)

print("Saved comparison figures to:", FIGURES_DIR)

In [ ]:
# ============================================================
# 12. Create chart figures for the IEEE report
# ============================================================

def save_bar(metric, title, ylabel, filename):
    plt.figure(figsize=(10, 5))
    data = summary_df.copy()
    plt.bar(data["method"], data[metric])
    plt.title(title)
    plt.ylabel(ylabel)
    plt.xticks(rotation=20, ha="right")
    plt.grid(axis="y", alpha=0.25)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / filename, dpi=220, bbox_inches="tight")
    plt.show()
    plt.close()

save_bar("detection_success_rate", "Page Detection Success Rate by Method", "Success rate", "chart_detection_success.png")
save_bar("mean_background_std", "Background Uniformity by Method", "Background std, lower is better", "chart_background_std.png")
save_bar("mean_text_contrast", "Text-to-Background Contrast by Method", "Contrast, higher is better", "chart_text_contrast.png")
save_bar("mean_runtime_ms", "Average Runtime by Method", "Runtime, ms, lower is better", "chart_runtime.png")

if summary_df["mean_ocr_confidence"].notna().any():
    save_bar("mean_ocr_confidence", "OCR Confidence by Method", "OCR confidence, higher is better", "chart_ocr_confidence.png")

if summary_df["mean_ssim"].notna().any():
    save_bar("mean_ssim", "SSIM by Method", "SSIM, higher is better", "chart_ssim.png")

In [ ]:
# ============================================================
# 13. Ablation study
# ============================================================

def proposed_without_lab(image):
    """Ablation: hybrid detection, but no Lab normalization."""
    quad, debug = detect_proposed_quad(image)
    t0 = time.perf_counter()
    if quad is None:
        return {"success": False, "quad": None, "output_color": None, "output_bw": None, "runtime_ms": 0}
    warped = four_point_transform(image, quad)
    bw = adaptive_bw(warped)
    runtime_ms = (time.perf_counter() - t0) * 1000
    return {"success": True, "quad": quad, "output_color": warped, "output_bw": bw, "runtime_ms": runtime_ms}


def proposed_contour_only_with_lab(image):
    """Ablation: contour detector only, but with Lab normalization."""
    t0 = time.perf_counter()
    quad, debug = detect_contour_quad(image)
    if quad is None:
        return {"success": False, "quad": None, "output_color": None, "output_bw": None, "runtime_ms": 0}
    warped = four_point_transform(image, quad)
    color_out, L_norm, illum = color_scan_output(warped)
    bw = adaptive_bw(L_norm)
    runtime_ms = (time.perf_counter() - t0) * 1000
    return {"success": True, "quad": quad, "output_color": color_out, "output_bw": bw, "runtime_ms": runtime_ms}


ABLATIONS = {
    "Full Proposed Method": lambda img: run_pipeline(img, "Proposed: Hybrid + Lab Norm"),
    "Without Lab Normalization": proposed_without_lab,
    "Contour Only + Lab": proposed_contour_only_with_lab
}

ablation_rows = []

for image_path in image_paths:
    img = read_rgb(image_path)
    stem = image_path.stem

    gt_mask = None
    gt_clean = None
    if not USE_UPLOADED_IMAGES:
        gt_mask_path = SYNTH_DIR / f"{stem}_gt_mask.png"
        gt_clean_path = SYNTH_DIR / f"{stem}_gt_clean.png"
        if gt_mask_path.exists():
            gt_mask = cv2.imread(str(gt_mask_path), cv2.IMREAD_GRAYSCALE)
        if gt_clean_path.exists():
            gt_clean = read_rgb(gt_clean_path)

    for name, func in ABLATIONS.items():
        res = func(img)
        pred_mask = polygon_mask(img.shape, res["quad"]) if res["quad"] is not None else np.zeros(img.shape[:2], np.uint8)

        detection_iou = np.nan if gt_mask is None else iou_masks(gt_mask, pred_mask)

        psnr_value = np.nan
        ssim_value = np.nan
        if gt_clean is not None and res["success"]:
            color_resized = cv2.resize(res["output_color"], (gt_clean.shape[1], gt_clean.shape[0]))
            psnr_value = psnr(gt_clean, color_resized, data_range=255)
            ssim_value = ssim(
                cv2.cvtColor(gt_clean, cv2.COLOR_RGB2GRAY),
                cv2.cvtColor(color_resized, cv2.COLOR_RGB2GRAY),
                data_range=255
            )

        ablation_rows.append({
            "image": image_path.name,
            "ablation": name,
            "success": int(res["success"]),
            "detection_iou": detection_iou,
            "psnr": psnr_value,
            "ssim": ssim_value,
            "background_std_lower_better": background_uniformity(res["output_bw"]) if res["success"] else np.nan,
            "text_contrast_higher_better": text_background_contrast(res["output_bw"]) if res["success"] else np.nan,
            "ocr_confidence_higher_better": ocr_confidence(res["output_bw"]) if res["success"] else np.nan,
            "runtime_ms_lower_better": res["runtime_ms"]
        })

ablation_df = pd.DataFrame(ablation_rows)
ablation_summary = (
    ablation_df.groupby("ablation")
    .agg(
        success_rate=("success", "mean"),
        mean_detection_iou=("detection_iou", "mean"),
        mean_psnr=("psnr", "mean"),
        mean_ssim=("ssim", "mean"),
        mean_background_std=("background_std_lower_better", "mean"),
        mean_text_contrast=("text_contrast_higher_better", "mean"),
        mean_ocr_confidence=("ocr_confidence_higher_better", "mean"),
        mean_runtime_ms=("runtime_ms_lower_better", "mean")
    )
    .reset_index()
)

ablation_df.to_csv(TABLES_DIR / "ablation_per_image.csv", index=False)
ablation_summary.to_csv(TABLES_DIR / "ablation_summary.csv", index=False)

display(ablation_summary)

In [ ]:
# ============================================================
# 14. Save report-ready tables as Markdown and LaTeX
# ============================================================

def save_table_formats(df, name):
    markdown_path = TABLES_DIR / f"{name}.md"
    latex_path = TABLES_DIR / f"{name}.tex"

    with open(markdown_path, "w") as f:
        f.write(df.to_markdown(index=False))

    with open(latex_path, "w") as f:
        f.write(df.to_latex(index=False, float_format="%.4f"))

    print("Saved:", markdown_path)
    print("Saved:", latex_path)

rounded_summary = summary_df.copy()
for col in rounded_summary.select_dtypes(include=[np.number]).columns:
    rounded_summary[col] = rounded_summary[col].round(4)

rounded_ablation = ablation_summary.copy()
for col in rounded_ablation.select_dtypes(include=[np.number]).columns:
    rounded_ablation[col] = rounded_ablation[col].round(4)

save_table_formats(rounded_summary, "method_summary_table")
save_table_formats(rounded_ablation, "ablation_summary_table")

print("\nMethod summary table:")
display(rounded_summary)

print("\nAblation summary table:")
display(rounded_ablation)

In [ ]:
# ============================================================
# 15. Generate README text for GitHub
# ============================================================

readme_text = f"""# Smart Document Scanner

This repository contains the implementation for the Digital Image Processing final term project:

**Confidence-Guided Hybrid Document Scanning with Shadow-Aware Enhancement for Smartphone-Captured Documents**

## Goal

The goal is to convert smartphone-captured document photos into clean scan-like images by:
1. detecting the document region,
2. correcting perspective distortion,
3. reducing shadows and uneven lighting,
4. improving text readability.

## Compared methods

- Baseline A: Contour + Otsu
- Baseline B: Hough + Adaptive Threshold
- Baseline C: Adaptive Mask + Adaptive Threshold
- Proposed: Confidence-Guided Hybrid Detection + Lab Luminance Normalization

## Main outputs

After running the notebook, results are saved in:

- `results/tables/per_image_results.csv`
- `results/tables/method_summary.csv`
- `results/tables/ablation_summary.csv`
- `results/figures/`
- `results/processed_outputs/`

## Metrics

The notebook calculates:

- page detection success rate,
- IoU for synthetic images with ground-truth masks,
- PSNR and SSIM for synthetic images with clean references,
- background uniformity,
- text-to-background contrast,
- OCR confidence if Tesseract is installed,
- runtime in milliseconds.

## Reproducibility

The notebook uses a fixed random seed: `{SEED}`.

For final report results, real document photos can be uploaded as a ZIP file by setting:

```python
USE_UPLOADED_IMAGES = True
```

## Requirements

- Python 3
- OpenCV
- NumPy
- Pandas
- Matplotlib
- scikit-image
- optional: pytesseract + tesseract-ocr
"""

readme_path = PROJECT_DIR / "README.md"
with open(readme_path, "w") as f:
    f.write(readme_text)

print("Saved README:", readme_path)

In [ ]:
# ============================================================
# 16. Zip all final outputs for download
# ============================================================

zip_path = Path("/content/smart_document_scanner_results.zip")

if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", PROJECT_DIR)

print("Created ZIP:", zip_path)
print("You can download this file from the Colab file panel or with the next cell.")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print("Not running in Colab. ZIP is saved at:", zip_path)

## 17. What to insert into the final IEEE paper

After running the notebook, use these files:

### Tables

- `results/tables/method_summary.csv`
- `results/tables/ablation_summary.csv`
- `results/tables/method_summary_table.tex`
- `results/tables/ablation_summary_table.tex`

### Figures

- `results/figures/comparison_*.png`
- `results/figures/chart_detection_success.png`
- `results/figures/chart_background_std.png`
- `results/figures/chart_text_contrast.png`
- `results/figures/chart_runtime.png`
- `results/figures/chart_ssim.png`

### Text for Experimental Setup

The experiments compared three classical baselines with the proposed hybrid method under the same preprocessing and output resolution. Each method was evaluated using detection success, IoU, PSNR, SSIM, background uniformity, text-to-background contrast, OCR confidence when available, and average runtime. The results were exported to CSV files and used to generate report-ready tables and visual comparison figures.